In [ ]:
#load packages
import spatialdata as sd
from spatialdata_io import xenium
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq
import scanpy as sc
import decoupler as dc
import plotnine as p9
import liana as li
from liana.method import MistyData, genericMistyData, lrMistyData
from liana.method.sp import RandomForestModel, LinearModel, RobustLinearModel
import pandas as pd


In [ ]:
# read xenium files
xenium_path = "path/to/xenium/output/" 
sdata = xenium(xenium_path)
adata = sdata['table']

#polygon coordinates to add TMA ID
polygon_df = pd.read_csv('path/to/coordinate/file/file.csv',sep=',',
    #skiprows=3,  
    skipinitialspace=True)

tma_polygons = {}
for name, group in polygon_df.groupby("Selection"):
    coords = list(zip(group["X"], group["Y"]))
    polygon = Polygon(coords)
    tma_polygons[name] = polygon


cell_coords = adata.obsm['spatial']
cell_points = [Point(x, y) for x, y in cell_coords]

def find_region(point, polygon_dict):
    for region, poly in polygon_dict.items():
        if poly.contains(point):
            return region
    return "Unassigned"
# adding TMA ID information
adata.obs['TMA_ID'] = [find_region(pt, tma_polygons) for pt in cell_points]
adata.write_h5ad('TMA_ID_annotated_file.h5ad')
